In [5]:
from pathlib import Path
import importlib
import pandas as pd
import numpy as np

import core_processing
importlib.reload(core_processing)

from core_processing import process_excel_to_csv

<div class="alert alert-info">
    Paths and Mapping configuration
</div>

In [6]:
# Base directory of the core_processing module
base_dir = Path(core_processing.__file__).resolve().parent

# File names | only names, not full paths
input_xlsx_name = "Australia_Mataranka.xlsx"
sheet_name = "Sheet1"
output_csv_name = input_xlsx_name.replace(".xlsx", ".csv")


# Full paths for notebook-side visibility
input_xlsx_path = base_dir / "Input_formatted_xlsx" / input_xlsx_name
output_csv_path = base_dir / "Output_cleaned_csv" / output_csv_name

print("Input Excel file:", input_xlsx_path)
print("Output CSV will be written to:", output_csv_path)


# Mapping: normalized Excel column names -> global schema attributes
# Left side HAS TO BE lowercase and can only feature "_" oder "-" as special characters
# Right side HAS TO BE global schema attributes
mapping_db = {
    "bore_spring": "location",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "ca_mg-l": "Ca_in_mmol/L",
    "mg_mg-l": "Mg_in_mmol/L",
    "k_mg-l": "K_in_mmol/L",
    "na_mg-l": "Na_in_mmol/L",
    "cl_mg-l": "Cl_in_mmol/L",
    "so4_mg-l": "SO4_in_mmol/L",
    "alkalinity_meq-l": "HCO3_in_mmol/L",
}

Input Excel file: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\Zusatz_Python-Auslagerung\Input_formatted_xlsx\Australia_Mataranka.xlsx
Output CSV will be written to: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\Zusatz_Python-Auslagerung\Output_cleaned_csv\Australia_Mataranka.csv


<div class="alert alert-warning">
    <b>Special Case adjustments</b><br>
    - Only needed if data is not in correct format!<br>
    - Can be left empty, if every chemical paramter has mg/L format and if coordinates are in WSG_84 format, ...
</div>

In [7]:
df = pd.read_excel(input_xlsx_path, sheet_name=sheet_name)

# must match Excel exactly
col = "Alkalinity_meq-L"

if col in df.columns:
    # Convert values to numeric (handles <, >, N/D, strings, etc.)
    vals_meq = pd.to_numeric(df[col], errors="coerce")

    # Molar mass for converting meq/L to mg/L
    M_HCO3 = core_processing.MOLAR_MASS["HCO3"]  # 61.0168 g/mol

    # Convert in-place: mg/L = meq/L * molar mass
    df[col] = vals_meq * M_HCO3

    # Save back to the same file
    df.to_excel(input_xlsx_path, sheet_name=sheet_name, index=False)

    print(f"Column '{col}' converted from meq/L to mg/L (name unchanged).")
    print("Excel file overwritten:", input_xlsx_path)

else:
    print(f"Column '{col}' not found – no conversion applied.")

Column 'Alkalinity_meq-L' converted from meq/L to mg/L (name unchanged).
Excel file overwritten: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\Zusatz_Python-Auslagerung\Input_formatted_xlsx\Australia_Mataranka.xlsx


<div class="alert alert-info">
    Run main ecxel to csv processing
</div>

In [8]:
output_path = process_excel_to_csv(
    input_xlsx=input_xlsx_name,
    output_csv=output_csv_name,
    sheet_name=sheet_name,
    mapping=mapping_db,
)

print("Final CSV written to:", output_path)

Final CSV written to: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\Zusatz_Python-Auslagerung\Output_cleaned_csv\Australia_Mataranka.csv
